# Baseline Model Comparison

Compare our Multi-Agent model (v3) against traditional ML baselines:
- Logistic Regression
- Random Forest
- XGBoost
- Simple MLP

**Goal**: Validate that multi-agent architecture provides value over simpler methods.

In [ ]:
# Install dependencies
!pip install -q xgboost lightgbm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

print("Libraries loaded successfully!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
DATA_PATH = "/content/drive/MyDrive/Sepsis/data/processed/mimic_harmonized"
OUTPUT_PATH = "/content/drive/MyDrive/Sepsis/models/baseline_comparison"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Use same random seed as v3 for fair comparison
RANDOM_SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.1

# Features (same as multi-agent model)
VITALS_FEATURES = ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat']
LABS_FEATURES = ['bun', 'chloride', 'creatinine', 'wbc', 'bicarbonate', 'platelets',
                 'magnesium', 'calcium', 'potassium', 'sodium', 'glucose',
                 'fio2', 'ph', 'paco2', 'pao2', 'lactate', 'bilirubin']
ALL_FEATURES = VITALS_FEATURES + LABS_FEATURES

print(f"Features: {len(ALL_FEATURES)}")
print(f"Output: {OUTPUT_PATH}")

## 1. Load Data

In [ ]:
# Load processed data (same as v3)
print("Loading data...")
df = pd.read_hdf(f"{DATA_PATH}/mimic_processed_large.h5", key='data')

print(f"\nDataset loaded:")
print(f"  Total observations: {len(df):,}")
print(f"  Unique patients: {df['subject_id'].nunique():,}")
print(f"  Sepsis prevalence: {df['sepsis_label'].mean()*100:.1f}%")

## 2. Prepare Data for Baselines

For traditional ML models, we need to flatten the temporal data into single feature vectors per observation.

We'll use:
1. **Current values** - the observation at that time point
2. **Aggregated features** - mean, std, min, max over patient history (optional enhancement)

In [ ]:
# Prepare feature matrix
print("Preparing features...")

# Use current observation values
X = df[ALL_FEATURES].copy()
y = df['sepsis_label'].values
patient_ids = df['subject_id'].values

# Handle missing values - fill with median (simple approach for baselines)
X = X.fillna(X.median())

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution: {np.bincount(y.astype(int))}")

In [ ]:
# Patient-level split (SAME as multi-agent model for fair comparison)
print("\nSplitting data (patient-level)...")

unique_patients = df['subject_id'].unique()
np.random.seed(RANDOM_SEED)
np.random.shuffle(unique_patients)

n_test = int(len(unique_patients) * TEST_SIZE)
n_val = int(len(unique_patients) * VAL_SIZE)

test_patients = set(unique_patients[:n_test])
val_patients = set(unique_patients[n_test:n_test+n_val])
train_patients = set(unique_patients[n_test+n_val:])

# Create masks
train_mask = df['subject_id'].isin(train_patients)
val_mask = df['subject_id'].isin(val_patients)
test_mask = df['subject_id'].isin(test_patients)

# Split data
X_train, y_train = X[train_mask].values, y[train_mask]
X_val, y_val = X[val_mask].values, y[val_mask]
X_test, y_test = X[test_mask].values, y[test_mask]

print(f"Train: {len(X_train):,} observations, {len(train_patients)} patients")
print(f"Val: {len(X_val):,} observations, {len(val_patients)} patients")
print(f"Test: {len(X_test):,} observations, {len(test_patients)} patients")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features standardized")

## 3. Train Baseline Models

In [ ]:
# Store results
results = {}

def evaluate_model(name, model, X_test, y_test):
    """Evaluate model and return metrics"""
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    auroc = roc_auc_score(y_test, y_pred_proba)
    auprc = average_precision_score(y_test, y_pred_proba)
    
    results[name] = {
        'auroc': auroc,
        'auprc': auprc,
        'y_pred_proba': y_pred_proba,
        'y_pred': y_pred
    }
    
    print(f"  AUROC: {auroc:.4f}")
    print(f"  AUPRC: {auprc:.4f}")
    
    return auroc, auprc

In [ ]:
# 1. Logistic Regression
print("="*60)
print("Training Logistic Regression...")
print("="*60)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    class_weight='balanced'
)
lr_model.fit(X_train_scaled, y_train)
evaluate_model('Logistic Regression', lr_model, X_test_scaled, y_test)

In [ ]:
# 2. Random Forest
print("="*60)
print("Training Random Forest...")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=RANDOM_SEED,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)  # RF doesn't need scaling
evaluate_model('Random Forest', rf_model, X_test, y_test)

In [ ]:
# 3. XGBoost
print("="*60)
print("Training XGBoost...")
print("="*60)

# Calculate scale_pos_weight for imbalanced classes
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    use_label_encoder=False,
    eval_metric='auc'
)
xgb_model.fit(X_train, y_train)  # XGBoost doesn't need scaling
evaluate_model('XGBoost', xgb_model, X_test, y_test)

In [ ]:
# 4. Simple MLP (Neural Network)
print("="*60)
print("Training Simple MLP...")
print("="*60)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=RANDOM_SEED,
    early_stopping=True,
    validation_fraction=0.1
)
mlp_model.fit(X_train_scaled, y_train)
evaluate_model('Simple MLP', mlp_model, X_test_scaled, y_test)

In [ ]:
# Add Multi-Agent v3 results for comparison
results['Multi-Agent (v3)'] = {
    'auroc': 0.7263,
    'auprc': 0.6536,
    'y_pred_proba': None,  # We don't have these saved
    'y_pred': None
}

print("\nMulti-Agent (v3) results added for comparison")

## 4. Compare Results

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame([
    {'Model': name, 'AUROC': r['auroc'], 'AUPRC': r['auprc']}
    for name, r in results.items()
]).sort_values('AUROC', ascending=False)

print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot - AUROC
colors = ['#e74c3c' if 'Multi-Agent' in m else '#3498db' for m in comparison_df['Model']]
ax1 = axes[0]
bars = ax1.barh(comparison_df['Model'], comparison_df['AUROC'], color=colors)
ax1.set_xlabel('AUROC', fontsize=12)
ax1.set_title('AUROC Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim(0.5, 0.85)
for i, (bar, val) in enumerate(zip(bars, comparison_df['AUROC'])):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
             va='center', fontsize=10)

# Bar plot - AUPRC
ax2 = axes[1]
bars = ax2.barh(comparison_df['Model'], comparison_df['AUPRC'], color=colors)
ax2.set_xlabel('AUPRC', fontsize=12)
ax2.set_title('AUPRC Comparison', fontsize=14, fontweight='bold')
ax2.set_xlim(0.4, 0.8)
for i, (bar, val) in enumerate(zip(bars, comparison_df['AUPRC'])):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
             va='center', fontsize=10)

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to {OUTPUT_PATH}/model_comparison.png")

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
for name, r in results.items():
    if r['y_pred_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, r['y_pred_proba'])
        ax1.plot(fpr, tpr, label=f"{name} (AUC={r['auroc']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR Curve
ax2 = axes[1]
for name, r in results.items():
    if r['y_pred_proba'] is not None:
        precision, recall, _ = precision_recall_curve(y_test, r['y_pred_proba'])
        ax2.plot(recall, precision, label=f"{name} (AP={r['auprc']:.3f})")

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/roc_pr_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance (XGBoost)

In [ ]:
# XGBoost Feature Importance
feature_importance = pd.DataFrame({
    'feature': ALL_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if f in VITALS_FEATURES else '#3498db' for f in feature_importance['feature']]
plt.barh(feature_importance['feature'], feature_importance['importance'], color=colors)
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance\n(Red=Vitals, Blue=Labs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Features:")
print(feature_importance.tail(10).to_string(index=False))

## 6. Summary & Conclusions

In [ ]:
# Find best baseline
baseline_models = {k: v for k, v in results.items() if 'Multi-Agent' not in k}
best_baseline_name = max(baseline_models, key=lambda x: baseline_models[x]['auroc'])
best_baseline = baseline_models[best_baseline_name]

multi_agent = results['Multi-Agent (v3)']

print("="*70)
print("FINAL COMPARISON SUMMARY")
print("="*70)
print(f"\nBest Baseline: {best_baseline_name}")
print(f"  AUROC: {best_baseline['auroc']:.4f}")
print(f"  AUPRC: {best_baseline['auprc']:.4f}")
print(f"\nMulti-Agent (v3):")
print(f"  AUROC: {multi_agent['auroc']:.4f}")
print(f"  AUPRC: {multi_agent['auprc']:.4f}")
print(f"\nDifference (Multi-Agent - Best Baseline):")
auroc_diff = multi_agent['auroc'] - best_baseline['auroc']
auprc_diff = multi_agent['auprc'] - best_baseline['auprc']
print(f"  AUROC: {auroc_diff:+.4f} ({'better' if auroc_diff > 0 else 'worse'})")
print(f"  AUPRC: {auprc_diff:+.4f} ({'better' if auprc_diff > 0 else 'worse'})")
print("="*70)

if auroc_diff > 0.02:
    conclusion = "Multi-Agent architecture provides MEANINGFUL improvement over baselines."
elif auroc_diff > 0:
    conclusion = "Multi-Agent architecture provides MARGINAL improvement over baselines."
else:
    conclusion = "Baselines OUTPERFORM Multi-Agent architecture. Consider using simpler model."

print(f"\nCONCLUSION: {conclusion}")

In [ ]:
# Save results
save_results = {
    'comparison_date': datetime.now().isoformat(),
    'dataset': 'mimic_processed_large.h5',
    'n_test_samples': len(y_test),
    'results': {
        name: {'auroc': r['auroc'], 'auprc': r['auprc']}
        for name, r in results.items()
    },
    'best_baseline': best_baseline_name,
    'conclusion': conclusion
}

with open(f"{OUTPUT_PATH}/comparison_results.json", 'w') as f:
    json.dump(save_results, f, indent=2)

print(f"\nResults saved to {OUTPUT_PATH}/comparison_results.json")

In [ ]:
# Final comparison table for documentation
print("\n" + "="*70)
print("COPY THIS TABLE TO DOCUMENTATION")
print("="*70)
print("\n| Model | AUROC | AUPRC | Notes |")
print("|-------|-------|-------|-------|")
for _, row in comparison_df.iterrows():
    note = "**Best Overall**" if 'Multi-Agent' in row['Model'] and auroc_diff > 0 else ""
    if row['Model'] == best_baseline_name:
        note = "Best Baseline"
    print(f"| {row['Model']} | {row['AUROC']:.4f} | {row['AUPRC']:.4f} | {note} |")